# DNS Record Analysis

Investigates `text/dns` records across all 15 target domains.

Key question: **why does ed.gov account for ~94% of all DNS records (~24k vs <100 for every other domain)?**

Sections:
1. DNS records per domain (overall)
2. DNS records by domain × year
3. DNS records vs total captures — ratio analysis
4. ed.gov deep dive

In [ ]:
import sys
sys.path.insert(0, '..')

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

from config import TARGET_DOMAINS, DATA_DIR
from split_domains import domain_folder_name

sns.set_theme(style='whitegrid')

DOMAINS_DIR = Path('..') / DATA_DIR / 'domains'

def db_path(domain: str) -> Path:
    folder = domain_folder_name(domain, TARGET_DOMAINS)
    return DOMAINS_DIR / folder / 'cdxj.duckdb'

# Verify all DBs are accessible
missing = [d for d in TARGET_DOMAINS if not db_path(d).exists()]
print(f'{len(TARGET_DOMAINS) - len(missing)}/{len(TARGET_DOMAINS)} domain DBs found')
if missing:
    print('Missing:', missing)

## 1. DNS Records per Domain

In [ ]:
rows = []
for domain in TARGET_DOMAINS:
    p = db_path(domain)
    if not p.exists():
        continue
    con = duckdb.connect(str(p), read_only=True)
    dns_n   = con.sql("SELECT COUNT(*) FROM eot_captures WHERE mime = 'text/dns'").fetchone()[0]
    total_n = con.sql("SELECT COUNT(*) FROM eot_captures").fetchone()[0]
    con.close()
    rows.append({'domain': domain, 'dns_records': dns_n, 'total_records': total_n})

df = pd.DataFrame(rows).sort_values('dns_records', ascending=False)
df['dns_pct'] = (100 * df['dns_records'] / df['total_records']).round(4)
df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if d == 'ed.gov' else '#3498db' for d in df['domain']]
bars = ax.barh(df['domain'][::-1], df['dns_records'][::-1], color=colors[::-1])

for bar in bars:
    w = bar.get_width()
    if w > 0:
        ax.text(w + 50, bar.get_y() + bar.get_height() / 2,
                f'{w:,}', va='center', fontsize=9)

ax.set_xlabel('DNS Records (text/dns)')
ax.set_title('DNS Records per Domain\n(red = outlier)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# Same chart excluding ed.gov to see the other 14 domains clearly
df_no_ed = df[df['domain'] != 'ed.gov'].sort_values('dns_records', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_no_ed['domain'][::-1], df_no_ed['dns_records'][::-1], color='#3498db')

for bar in ax.patches:
    w = bar.get_width()
    if w > 0:
        ax.text(w + 2, bar.get_y() + bar.get_height() / 2,
                f'{w:,}', va='center', fontsize=9)

ax.set_xlabel('DNS Records (text/dns)')
ax.set_title('DNS Records per Domain — ed.gov excluded')
plt.tight_layout()
plt.show()

## 2. DNS Records by Domain × Year

In [ ]:
year_rows = []
for domain in TARGET_DOMAINS:
    p = db_path(domain)
    if not p.exists():
        continue
    con = duckdb.connect(str(p), read_only=True)
    results = con.sql("""
        SELECT crawl_year, COUNT(*) AS dns_records
        FROM eot_captures
        WHERE mime = 'text/dns'
        GROUP BY crawl_year
        ORDER BY crawl_year
    """).fetchall()
    con.close()
    for crawl_year, count in results:
        year_rows.append({'domain': domain, 'crawl_year': int(crawl_year), 'dns_records': count})

df_year = pd.DataFrame(year_rows)
df_year

In [ ]:
# Heatmap: domain × year
pivot = df_year.pivot(index='domain', columns='crawl_year', values='dns_records').fillna(0).astype(int)
# Sort by total DNS records desc
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('DNS Records: Domain × Crawl Year')

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        label = f'{val:,}' if val > 0 else ''
        ax.text(j, i, label, ha='center', va='center', fontsize=8)

plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Same heatmap, ed.gov excluded so other domains' variation is visible
pivot_no_ed = pivot.drop(index='ed.gov', errors='ignore')

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(pivot_no_ed.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(pivot_no_ed.columns)))
ax.set_xticklabels(pivot_no_ed.columns)
ax.set_yticks(range(len(pivot_no_ed.index)))
ax.set_yticklabels(pivot_no_ed.index)
ax.set_title('DNS Records: Domain × Crawl Year — ed.gov excluded')

for i in range(len(pivot_no_ed.index)):
    for j in range(len(pivot_no_ed.columns)):
        val = pivot_no_ed.values[i, j]
        label = f'{val:,}' if val > 0 else ''
        ax.text(j, i, label, ha='center', va='center', fontsize=8)

plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

## 3. DNS Records vs Total Captures — Ratio Analysis

How many DNS records does each domain have relative to its total capture count?
A high ratio suggests unusual crawler behavior or DNS pre-fetching.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

df_plot = df.sort_values('total_records', ascending=False)

# Left: total records vs DNS records (scatter, log scale)
ax = axes[0]
for _, row in df_plot.iterrows():
    color = '#e74c3c' if row['domain'] == 'ed.gov' else '#3498db'
    ax.scatter(row['total_records'], row['dns_records'], color=color, s=80, zorder=3)
    ax.annotate(row['domain'], (row['total_records'], row['dns_records']),
                textcoords='offset points', xytext=(5, 3), fontsize=7)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Total Captures (log scale)')
ax.set_ylabel('DNS Records (log scale)')
ax.set_title('Total Captures vs DNS Records')

# Right: DNS records as % of total
ax2 = axes[1]
df_pct = df.sort_values('dns_pct', ascending=True)
colors2 = ['#e74c3c' if d == 'ed.gov' else '#3498db' for d in df_pct['domain']]
ax2.barh(df_pct['domain'], df_pct['dns_pct'], color=colors2)
for bar in ax2.patches:
    w = bar.get_width()
    if w > 0:
        ax2.text(w + 0.0001, bar.get_y() + bar.get_height() / 2,
                 f'{w:.4f}%', va='center', fontsize=8)
ax2.set_xlabel('DNS Records as % of Total Captures')
ax2.set_title('DNS Share of Total Captures per Domain')

plt.tight_layout()
plt.show()

print('\nSummary table:')
print(df[['domain','dns_records','total_records','dns_pct']].sort_values('dns_pct', ascending=False).to_string(index=False))

## 4. ed.gov Deep Dive

What exactly are these DNS records? Sample rows and breakdown by year.

In [ ]:
ed_con = duckdb.connect(str(db_path('ed.gov')), read_only=True)

# Sample DNS rows
sample = ed_con.sql("""
    SELECT url, host, surtkey, fetch_timestamp, crawl_year, status, "length"
    FROM eot_captures
    WHERE mime = 'text/dns'
    ORDER BY crawl_year, fetch_timestamp
    LIMIT 20
""").df()
sample

In [ ]:
# DNS records by year for ed.gov
ed_by_year = ed_con.sql("""
    SELECT
        crawl_year,
        COUNT(*) AS dns_records,
        COUNT(DISTINCT url) AS unique_dns_urls,
        COUNT(DISTINCT host) AS unique_hosts
    FROM eot_captures
    WHERE mime = 'text/dns'
    GROUP BY crawl_year
    ORDER BY crawl_year
""").df()
print('ed.gov DNS by year:')
ed_by_year

In [ ]:
# ed.gov: total captures vs DNS records by year
ed_total_by_year = ed_con.sql("""
    SELECT crawl_year, COUNT(*) AS total_records
    FROM eot_captures
    GROUP BY crawl_year
    ORDER BY crawl_year
""").df()

ed_compare = ed_total_by_year.merge(ed_by_year[['crawl_year','dns_records']], on='crawl_year', how='left').fillna(0)
ed_compare['dns_records'] = ed_compare['dns_records'].astype(int)
ed_compare['dns_pct'] = (100 * ed_compare['dns_records'] / ed_compare['total_records']).round(4)

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(ed_compare))
width = 0.35
ax.bar([i - width/2 for i in x], ed_compare['total_records'], width, label='Total captures', color='#3498db')
ax.bar([i + width/2 for i in x], ed_compare['dns_records'], width, label='DNS records', color='#e74c3c')
ax.set_xticks(list(x))
ax.set_xticklabels(ed_compare['crawl_year'])
ax.set_ylabel('Record count')
ax.set_title('ed.gov — Total Captures vs DNS Records by Year')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e6:.1f}M' if v >= 1e6 else f'{v:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

print(ed_compare.to_string(index=False))

In [ ]:
# Most common hosts in ed.gov DNS records
ed_con.sql("""
    SELECT host, crawl_year, COUNT(*) AS n
    FROM eot_captures
    WHERE mime = 'text/dns'
    GROUP BY host, crawl_year
    ORDER BY n DESC
    LIMIT 30
""").df()

In [ ]:
ed_con.close()